# Bridge: OSMnx/iduedu → Arctic Format

Перевод графа из OSMnx или iduedu в формат arctic (transport_df, G для make_g / provision).

Позволяет использовать Arctic на уровне кварталов: блоки + OSM-граф → bridge → make_g → provision.

In [8]:
import sys
from pathlib import Path

ROOT = Path(".").resolve()
if ROOT.name == "bridge":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# arctic_access scripts path
ARCTIC_PATH = ROOT / "arctic_access"
if str(ARCTIC_PATH) not in sys.path:
    sys.path.insert(0, str(ARCTIC_PATH))

import geopandas as gpd
import pandas as pd

## 1. Загрузка блоков (кварталов)

Используем данные arctic_access: берём 3 ближайших поселения к центроиду — маленький кусок для быстрой загрузки OSM.

In [9]:
from scripts.preprocesser.preprocesser import get_data
from scripts.calculator.calculator_this_pipeline import make_block_scheme
from scripts.preprocesser.constants import (
    transport_mode_name_mapper,
    transport_modes,
    MERCATOR_CRS,
)

DATA_PATH = str(ROOT / "arctic_access" / "data") + "/"
SETTL_NAME = "mezen"
SERVICE_NAME = "health"

settl, df_service, _, _ = get_data(
    DATA_PATH,
    SETTL_NAME,
    transport_mode_name_mapper,
    transport_modes,
    SERVICE_NAME,
)

# Маленький кусок: 3 ближайших поселения к центроиду (быстрый OSM)
centroid = settl.geometry.union_all().centroid
settl["_dist"] = settl.geometry.centroid.distance(centroid)
settl_sub = settl[settl["_dist"] < 100_000].drop(columns=["_dist"])
blocks_gdf = make_block_scheme(settl_sub, df_service, service_name=SERVICE_NAME)
boundary = settl_sub.geometry.union_all()
boundary_wgs84 = gpd.GeoSeries([boundary], crs=settl.crs).to_crs(4326).iloc[0]

print(f"Блоков: {len(blocks_gdf)} ({', '.join(blocks_gdf['name'].tolist())})")
blocks_gdf.head()

Блоков: 24 (Bereznik, Bor, Bych'e, Dorogorskoe, Zherd', Zhukova, Zaakakur'e, Zaozer'e, Zaton, Kar'epol'e, Kil'tsa, Kimzha, Koz'mogorodskoe, Lampozhnja, Mezen', Morozilka, Petrova, Petrovka, Pechische, Pogorelets, Timoschel'e, Ust'-Njafta, Ust'-Peza, Chetsa)


,id,name,geometry,population,epsg,capacity_health
0,2,Bereznik,"POLYGON ((5009611.208 9735768.885, 5009606.392...",134,3857,600
1,3,Bor,"POLYGON ((4941825.43 9798870.626, 4941820.615 ...",120,3857,0
2,4,Bych'e,"POLYGON ((5016216.906 9819902.177, 5016212.091...",146,3857,600
3,6,Dorogorskoe,"POLYGON ((4954863.169 9769048.737, 4954858.354...",349,3857,600
4,9,Zherd',"POLYGON ((4976458.037 9760578.007, 4976453.222...",107,3857,1200


## 2. Drive-граф (iduedu или OSMnx)

In [10]:
try:
    from iduedu import get_drive_graph
    G_drive = get_drive_graph(territory=boundary_wgs84)
    use_iduedu = True
    print("G_drive из iduedu")
except ImportError:
    import osmnx as ox
    from bridge import ensure_graph_has_time_min
    try:
        G_drive = ox.graph_from_polygon(boundary_wgs84, network_type="drive")
    except (RuntimeError, IndexError):
        # Полярные регионы: UTM может не определяться — используем bbox (west, south, east, north)
        minx, miny, maxx, maxy = gpd.GeoSeries([boundary_wgs84], crs=4326).total_bounds
        bbox = (minx, miny, maxx, maxy)
        G_drive = ox.graph_from_bbox(bbox=bbox, network_type="drive")
    G_drive = ensure_graph_has_time_min(G_drive)
    use_iduedu = False
    print("G_drive из OSMnx")

print(f"Узлов: {G_drive.number_of_nodes()}, рёбер: {G_drive.number_of_edges()}")

2026-03-14 13:44:04.169 | INFO     | Downloading drive network via Overpass ...
2026-03-14 13:44:04.176 | INFO     | Downloading network via Overpass done!
2026-03-14 13:44:04.302 | WARNING  | Removing 16 nodes from 1 smaller strongly connected components. These are subgraphs where nodes are internally reachable but isolated from the rest. Retaining only the largest strongly connected component (541 nodes).


G_drive из iduedu
Узлов: 541, рёбер: 1434


## 3. Bridge: graph_to_arctic_format

In [11]:
from bridge import graph_to_arctic_format, settl_from_blocks

transport_df, G_arctic = graph_to_arctic_format(
    blocks_gdf,
    G_drive,
    G_transit=None,
    service_name=SERVICE_NAME,
    modalities=["drive"],
    mode_mapping=None,  # опционально: {"drive": "Regular road"} для Arctic
    use_iduedu=use_iduedu,
)

settl = settl_from_blocks(blocks_gdf)

print(f"transport_df: {len(transport_df)} рёбер")
transport_df.head(10)

transport_df: 546 рёбер


,edge1,edge2,drive
0,Bereznik,Bor,51.24
1,Bereznik,Bych'e,55.55
2,Bereznik,Dorogorskoe,37.78
3,Bereznik,Zherd',22.65
4,Bereznik,Zhukova,20.29
5,Bereznik,Zaakakur'e,51.03
6,Bereznik,Zaozer'e,43.06
7,Bereznik,Zaton,95.92
8,Bereznik,Kar'epol'e,395.91
9,Bereznik,Kil'tsa,48.92


## 4. make_g → Arctic граф

In [12]:
from scripts.preprocesser.gcreator import make_g

G_undirected = make_g(transport_df, ["drive"], blocks_gdf, settl)

print(f"G_undirected: {G_undirected.number_of_nodes()} узлов, {G_undirected.number_of_edges()} рёбер")

G_undirected: 24 узлов, 1092 рёбер


In [13]:
G_undirected.edges(data=True)

MultiEdgeDataView([('Bereznik', 'Bor', {'weight': 51.24, 'label': 'drive'}), ('Bereznik', 'Bor', {'weight': 51.24, 'label': 'drive'}), ('Bereznik', 'Bor', {'weight': 51.24, 'label': 'drive'}), ('Bereznik', 'Bor', {'weight': 51.24, 'label': 'drive'}), ('Bereznik', "Bych'e", {'weight': 55.55, 'label': 'drive'}), ('Bereznik', "Bych'e", {'weight': 55.55, 'label': 'drive'}), ('Bereznik', "Bych'e", {'weight': 55.55, 'label': 'drive'}), ('Bereznik', "Bych'e", {'weight': 55.55, 'label': 'drive'}), ('Bereznik', 'Dorogorskoe', {'weight': 37.78, 'label': 'drive'}), ('Bereznik', 'Dorogorskoe', {'weight': 37.78, 'label': 'drive'}), ('Bereznik', 'Dorogorskoe', {'weight': 37.78, 'label': 'drive'}), ('Bereznik', 'Dorogorskoe', {'weight': 37.78, 'label': 'drive'}), ('Bereznik', "Zherd'", {'weight': 22.65, 'label': 'drive'}), ('Bereznik', "Zherd'", {'weight': 22.65, 'label': 'drive'}), ('Bereznik', "Zherd'", {'weight': 22.65, 'label': 'drive'}), ('Bereznik', "Zherd'", {'weight': 22.65, 'label': 'drive'}

## 5. Расчёт провижена (Arctic)

In [14]:
from scripts.preprocesser.constants import CONST_BASE_DEMAND, service_radius_minutes
import scripts.model.provision as provision

radius = service_radius_minutes.get(SETTL_NAME, 60)

G_with_prov, prov_result, _ = provision.calculate_graph_provision(
    G_undirected,
    service_radius=radius,
    const_base_demand=CONST_BASE_DEMAND,
    service_name=SERVICE_NAME,
    return_assignment=True,
    method="lp",
)

print(prov_result.reset_index()[["name", "provision", "demand", f"capacity_{SERVICE_NAME}"]].to_string())

               name  provision  demand  capacity_health
0          Bereznik        1.0    17.0            600.0
1               Bor        1.0    15.0              0.0
2            Bych'e        1.0    18.0            600.0
3       Dorogorskoe        1.0    42.0            600.0
4            Zherd'        1.0    13.0           1200.0
5           Zhukova        1.0    15.0              0.0
6        Zaakakur'e        1.0     9.0            600.0
7          Zaozer'e        1.0     4.0            600.0
8             Zaton        1.0    15.0              0.0
9        Kar'epol'e        1.0     8.0            600.0
10          Kil'tsa        1.0    10.0            600.0
11           Kimzha        1.0     9.0            600.0
12  Koz'mogorodskoe        1.0    10.0            600.0
13       Lampozhnja        1.0     9.0            600.0
14           Mezen'        1.0   340.0           1800.0
15        Morozilka        1.0    15.0              0.0
16          Petrova        1.0     1.0          